# Business Understanding

**Proyecto:** NBA ML Predictor  
**Temporada:** 2025-26 Regular Season  
**Metodología:** CRISP-DM  



## Contexto del Problema

La NBA (National Basketball Association) es la liga de baloncesto profesional más importante del mundo. Cada temporada regular comprende 82 partidos por equipo, distribuidos entre 30 franquicias, lo que genera un volumen de datos estadísticos considerable y estructurado que la hace especialmente adecuada para análisis cuantitativo.

El baloncesto es un deporte de alta densidad estadística: cada partido produce cientos de métricas individuales y colectivas (puntos, rebotes, asistencias, eficiencia de tiro, diferencial de puntos, entre otras). Esta granularidad lo diferencia de otros deportes y facilita la construcción de modelos predictivos con señal real.

**Por qué es un dominio relevante para Machine Learning:**

- Los datos son públicos, oficiales y están bien estructurados.
- Existe continuidad temporal clara: el historial de un equipo o jugador es directamente interpretable como secuencia.
- Hay múltiples niveles de análisis: jugador individual, equipo, partido, temporada.
- Los patrones de rendimiento son parcialmente estables pero contienen varianza real, lo que hace que el problema no sea trivial.

**Valor de negocio de predecir resultados deportivos:**

| Sector | Aplicación |
|---|---|
| Apuestas deportivas | Ajuste de líneas y odds con modelos propios |
| Cuerpos técnicos | Análisis de rivales y preparación táctica |
| Scouting y front office | Evaluación de impacto de jugadores en resultados |
| Medios y contenido | Narrativas predictivas previas a cada jornada |
| Fantasy sports | Selección de jugadores basada en proyecciones estadísticas |


## 2. Definición Formal de los Problemas

### Problema 1 — Clasificación Binaria: Predicción de Resultado de Partido

**Pregunta de negocio:** Dado el historial reciente de un equipo, ¿ganará su próximo partido?

**Definición formal:**

Sea $X_t$ el vector de features construido a partir de los partidos anteriores al partido $t$ de un equipo. El modelo aprende una función:

$$f: X_t \rightarrow \hat{y} \in \{0, 1\}$$

- **Variable objetivo:** `WL` — resultado del partido.
  - `1` = victoria
  - `0` = derrota
- **Tipo de problema:** clasificación binaria supervisada.
- **Unidad de observación:** un partido de un equipo específico.
- **Features:** métricas promedio del equipo en los últimos N partidos previos al que se predice.



### Problema 2 — Regresión: Predicción de Puntos de un Jugador

**Pregunta de negocio:** Dado el historial reciente de un jugador, ¿cuántos puntos anotará en su próximo partido?

**Definición formal:**

Sea $X_t$ el vector de features construido a partir de los partidos anteriores al partido $t$ de un jugador. El modelo aprende una función:

$$f: X_t \rightarrow \hat{y} \in \mathbb{R}^+$$

- **Variable objetivo:** `PTS` — puntos anotados en el partido.
- **Tipo de problema:** regresión supervisada.
- **Unidad de observación:** un partido de un jugador específico.
- **Features:** métricas promedio del jugador en los últimos N partidos previos al que se predice.

## 3. Justificación de los Datos

Los datos provienen de la **NBA Stats API** (stats.nba.com), la fuente oficial de estadísticas de la liga. Se consumen a través de la librería `nba_api` y se almacenan en una base de datos PostgreSQL local.

**Por qué estos datos son válidos para el problema:**

**Oficialidad:** Los datos son producidos directamente por la NBA. No son scraping de terceros ni estimaciones; reflejan exactamente lo que ocurrió en cada partido.

**Cobertura completa:** Se usa la temporada regular 2025-26 en su totalidad. Todos los equipos, todos los partidos de la temporada regular, sin filtros arbitrarios.

**Granularidad de partido a partido:** Los datos están disponibles a nivel de game log por equipo y por jugador. Esto permite construir features temporales (promedios móviles, tendencias recientes) que son más informativas que los agregados de temporada.

**Amplitud de métricas:** Las tablas incluyen tanto estadísticas de rendimiento individual (puntos, rebotes, asistencias, porcentajes de tiro) como colectivas (diferencial de puntos por partido, victorias/derrotas acumuladas, ritmo de juego).

**Tablas utilizadas:**

| Tabla | Contenido | Uso en el modelo |
|---|---|---|
| `fact_team_game_logs` | Un registro por partido por equipo | Features para clasificación |
| `fact_player_game_logs` | Un registro por partido por jugador | Features para regresión |
| `fact_team_season_stats` | Estadísticas agregadas por equipo | Referencia y validación |
| `fact_player_season_stats` | Estadísticas agregadas por jugador | Referencia y validación |
| `dim_teams` | Catálogo de equipos | Joins y contexto |
| `dim_players` | Catálogo de jugadores | Joins y contexto |

## 4. Criterios de Éxito del Modelo

### Clasificación — Predicción de Resultado de Partido

| Métrica | Descripción |
|---|---|
| **Accuracy** | Proporción de predicciones correctas sobre el total |
| **F1-Score** | Media armónica entre precisión y recall |
| **AUC-ROC** | Capacidad del modelo para separar victorias de derrotas a distintos umbrales |

**Por qué no basta con accuracy:**  
Si en un conjunto de prueba el 60% de los partidos son victorias locales, un clasificador trivial que siempre predice victoria alcanzaría un 60% de accuracy sin aprender ningún patrón real. El F1-Score penaliza este comportamiento al considerar falsos positivos y falsos negativos por separado. El AUC-ROC evalúa la calidad de la probabilidad predicha, independientemente del umbral de decisión.

**Umbral de referencia:** Un modelo competente debería superar el 60% de accuracy y un AUC-ROC superior a 0.62 en datos de validación.



### Regresión — Predicción de Puntos de un Jugador

| Métrica | Descripción | Interpretación en el dominio |
|---|---|---|
| **RMSE** | Raíz del error cuadrático medio | Error promedio en puntos; penaliza errores grandes |
| **MAE** | Error absoluto medio | Cuántos puntos se equivoca el modelo en promedio |
| **R²** | Coeficiente de determinación | Qué proporción de la varianza de PTS explica el modelo |

**Interpretación práctica:**  
Un MAE de 4 puntos significa que, en promedio, la predicción difiere 4 puntos del valor real. Para un jugador con promedio de 20 puntos, esto representa un error relativo del 20%, que es aceptable dada la varianza natural del juego. El R² indica si el modelo captura tendencias reales o simplemente predice la media del jugador.

## 5. Restricciones y Consideraciones

**Alcance de los datos:**
- Solo se utilizan datos de la temporada regular 2025-26.
- Los datos de playoffs quedan fuera del alcance. El playoff es una competencia distinta (eliminatoria, equipos distintos, motivación diferente) que introduciría distribuciones de rendimiento no comparables con los de temporada regular.

**Prevención de data leakage:**  
Esta es la restricción más crítica del proyecto. En ningún caso se puede incluir como feature información del partido que se está prediciendo. Toda la construcción de features debe basarse exclusivamente en partidos anteriores al partido objetivo.

- Correcto: promedio de puntos de los últimos 5 partidos anteriores al partido $t$.
- Incorrecto: estadísticas del partido $t$ mismo (minutos jugados, rivales del día, resultado final).

Este principio se aplicará en la fase de feature engineering y se verificará antes de cualquier entrenamiento.

**Otras restricciones:**
- No se modela el impacto de lesiones ni de bajas de último momento (datos no disponibles de forma estructurada).
- No se incorporan datos externos como condiciones de viaje, back-to-backs o altitud (posible extensión futura).
- Los modelos son por equipo o por jugador de forma individual, no multijugador simultáneo.

## 6. Flujo CRISP-DM Aplicado al Proyecto

CRISP-DM (Cross-Industry Standard Process for Data Mining) es la metodología que estructura este proyecto. Las seis fases no son estrictamente lineales; en la práctica implican iteraciones, especialmente entre las fases de modelado y evaluación.

| Fase | Nombre | Aplicación en este proyecto |
|---|---|---|
| 1 | **Business Understanding** | Definir los dos problemas predictivos, criterios de éxito y restricciones |
| 2 | **Data Understanding** | Explorar game logs de equipos y jugadores, verificar completitud, detectar outliers |
| 3 | **Data Preparation** | Construir features de ventana temporal (rolling averages), encodings, split temporal train/test |
| 4 | **Modeling** | Entrenar modelos de clasificación (equipo) y regresión (jugador): baseline, árbol, ensemble |
| 5 | **Evaluation** | Comparar modelos con métricas definidas, verificar ausencia de leakage, seleccionar modelo final |
| 6 | **Deployment** | Serializar modelos entrenados, construir pipeline de predicción sobre datos nuevos |

**Ciclo de iteración principal:**

```
Business Understanding
        |
        v
Data Understanding  <--------+
        |                    |
        v                    |
Data Preparation             |
        |                    |
        v                    |
    Modeling  ---------------+  (si los resultados indican que los features son insuficientes)
        |
        v
   Evaluation
        |
        v
   Deployment
```